In [1]:
from datetime import date

import hisepy
import os
import pandas as pd

In [2]:
if not os.path.isdir('output'):
    os.mkdir('output')

### Helper functions

In [3]:
def cache_uuid_path(uuid):
    hise_res = hisepy.reader.cache_files([uuid])
    cache_file = hise_res[0]

    return cache_file

In [4]:
def read_csv_uuid(csv_uuid, index_col = None):
    csv_file = cache_uuid_path(csv_uuid)
    if index_col is None:
        df = pd.read_csv(csv_file)
    else:
        df = pd.read_csv(csv_file, index_col = index_col)
    return df

In [5]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

### Retrieve DESeq2 results from HISE

In [6]:
deg_uuid = 'ae34daa7-00b3-4c4a-b770-c3922da030ef'
deg = read_csv_uuid(deg_uuid)

In [7]:
d0_d7_deg_uuid = 'bd2e9e22-f425-4927-b472-18d15711df27'
d0_d7_deg = read_csv_uuid(d0_d7_deg_uuid, index_col = 0)

In [8]:
deg = pd.concat([deg, d0_d7_deg], axis = 0)

In [9]:
deg['contrast'].value_counts()

contrast
sample.visitName         376678
cohort.cohortGuid        373942
CMV                      373942
subject.biologicalSex    373942
Name: count, dtype: int64

In [10]:
contrast_to_name = {
    'cohort.cohortGuid': 'Subject Age: Younger | Older',
    'CMV': 'CMV Status: Negative | Positive',
    'subject.biologicalSex': 'Subject Sex: Female | Male',
    'sample.visitName': 'Flu Vaccine: Day 0 | Day 7'
}

In [11]:
contrast_sets = {
    'cohort.cohortGuid': { 'fg': 'Older Adult', 'bg': 'Younger Adult' },
    'CMV': { 'fg': 'Positive', 'bg': 'Negative' },
    'subject.biologicalSex': {'fg': 'Male', 'bg': 'Female' },
    'sample.visitName': {'fg': 'Flu Year 1 Day 7', 'bg': 'Flu Year 1 Day 0'}
}

In [12]:
rename_dict = {
    'celltype': 'AIFI_L3',
    'log2FoldChange': 'log2fc'
}

In [13]:
contrasts = deg['contrast'].unique()

In [14]:
contrasts

array(['cohort.cohortGuid', 'CMV', 'subject.biologicalSex',
       'sample.visitName'], dtype=object)

In [15]:
contrasts = deg['contrast'].unique()
deg_dfs = {}
for contrast in contrasts:
    df = deg.loc[deg['contrast'] == contrast].copy()
    contrast_name = contrast_to_name[contrast]

    df['fg'] = contrast_sets[contrast]['fg']
    df['bg'] = contrast_sets[contrast]['bg']
    
    df = df.rename(rename_dict, axis = 1)
    df = df[['AIFI_L3', 'fg', 'bg', 
             'gene', 'log2fc', 
             'padj', 'pvalue', 'stat']]

    deg_dfs[contrast] = df

In [16]:
contrast_files = {
    'cohort.cohortGuid': 'diha_age-group_deseq2_results_{d}.csv'.format(d = date.today()),
    'CMV': 'diha_cmv-status_deseq2_results_{d}.csv'.format(d = date.today()),
    'subject.biologicalSex': 'diha_biological-sex_deseq2_results_{d}.csv'.format(d = date.today()),
    'sample.visitName': 'diha_flu-vaccine_deseq2_results_{d}.csv'.format(d = date.today())
}

In [17]:
out_files = []
for contrast in contrasts:
    deg_df = deg_dfs[contrast]
    print(deg_df.head())
    out_file = 'output/' + contrast_files[contrast]

    deg_df.to_csv(out_file, index = False)
    out_files.append(out_file)

  AIFI_L3           fg             bg        gene    log2fc      padj  \
0    ASDC  Older Adult  Younger Adult  AL669831.5  0.049667  0.999852   
1    ASDC  Older Adult  Younger Adult       NOC2L  0.098326  0.999852   
2    ASDC  Older Adult  Younger Adult       ISG15 -0.719866  0.999852   
3    ASDC  Older Adult  Younger Adult        SDF4  0.041011  0.999852   
4    ASDC  Older Adult  Younger Adult     B3GALT6 -0.583031  0.999852   

     pvalue      stat  
0  0.898642  0.127377  
1  0.685778  0.404592  
2  0.050618 -1.954706  
3  0.834365  0.209107  
4  0.062161 -1.865149  
     AIFI_L3        fg        bg        gene    log2fc      padj    pvalue  \
7582    ASDC  Positive  Negative  AL669831.5  0.310358  0.998107  0.426799   
7583    ASDC  Positive  Negative       NOC2L -0.137449  0.998107  0.572242   
7584    ASDC  Positive  Negative       ISG15  0.313304  0.998107  0.390219   
7585    ASDC  Positive  Negative        SDF4  0.126356  0.998107  0.518980   
7586    ASDC  Positive  Neg

## Upload DEG data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [18]:
study_space_uuid = 'de025812-5e73-4b3c-9c3b-6d0eac412f2a'
title = 'DIHA DEG results {d}'.format(d = date.today())

In [19]:
search_id = element_id()
search_id

'dubnium-silicon-radium'

In [20]:
in_files = [deg_uuid, d0_d7_deg_uuid]
in_files

['ae34daa7-00b3-4c4a-b770-c3922da030ef',
 'bd2e9e22-f425-4927-b472-18d15711df27']

In [21]:
out_files

['output/diha_age-group_deseq2_results_2025-02-06.csv',
 'output/diha_cmv-status_deseq2_results_2025-02-06.csv',
 'output/diha_biological-sex_deseq2_results_2025-02-06.csv',
 'output/diha_flu-vaccine_deseq2_results_2025-02-06.csv']

In [22]:
len(out_files)

4

In [23]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

Cannot determine the current notebook.
1) /home/workspace/sound-life-scrna-analysis/04-file-sets/Python_assemble_deg_results.ipynb
2) /home/workspace/sound-life-scrna-analysis/04-file-sets/output/Untitled.ipynb
3) /home/workspace/sound-life-scrna-analysis/templates-and-examples/TE-R_template_h5ad_data_per_sample.ipynb
Please select (1-3) 


 1


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '0cf9ad23-d50f-4b3f-8c50-3e32f94428b8',
 'ProcessId': 'cdaf3c76-34a8-4baa-9354-2d1bbf04850e',
 'WorkflowId': 'fe19c999-4634-46db-9557-811c6c730888',
 'FileIds': ['a1e7a505-50b9-49b0-bf51-45c056aaa234',
  '3fab6ec6-7bad-4548-93c4-08804e64f69e',
  '704aa861-593f-401d-9511-19e86a9a292f',
  '312f8f09-3c8c-429b-8a56-1f3b9c15f04b']}

In [24]:
import session_info
session_info.show()